# 5. Специализированные методы: Croston и вариации

**Сквозной вопрос:** как разделить прогноз на «будет ли ненулевой спрос» и «если да, то сколько».

В этом ноутбуке:
- Croston (1972) — раздельные SES для интервалов и размера;
- SBA (Syntetos–Boylan Approximation) — коррекция смещения Croston;
- TSB (Teunter–Syntetos–Babai) — вероятностный подход;
- ADIDA / IMAPA — агрегация + дезагрегация;
- сравнение всех методов по классам спроса.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import (
    Croston, CrostonOptimized, CrostonSBA, TSB,
    ADIDA, IMAPA
)

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные и хелперы

Используем те же M5 CA_1 данные и классификацию ADI/CV².

In [ ]:
# Загрузка
sales = pd.read_csv('../data/sales_train_validation.csv')
calendar = pd.read_csv('../data/calendar.csv')

sales_ca1 = sales[sales['store_id'] == 'CA_1'].copy()

id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_ca1.columns if c.startswith('d_')]

df_long = sales_ca1.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

df_long = df_long.merge(calendar[['d', 'date']], on='d', how='left')
df_long['date'] = pd.to_datetime(df_long['date'])
df_long = df_long.sort_values(['id', 'date']).reset_index(drop=True)

print(f"Товаров: {df_long['id'].nunique()}, период: {df_long['date'].min()} — {df_long['date'].max()}")

In [ ]:
# Классификация (из предыдущего ноутбука)
def classify_demand(series):
    nonzero_idx = np.where(series > 0)[0]

    if len(nonzero_idx) == 0:
        return {'adi': np.inf, 'cv2': np.inf, 'class': 'lumpy'}

    if len(nonzero_idx) == 1:
        adi = len(series)
        cv2 = 0
    else:
        intervals = np.diff(nonzero_idx)
        adi = intervals.mean() if len(intervals) > 0 else len(series)

        nonzero_vals = series[nonzero_idx]
        if nonzero_vals.mean() > 0:
            cv2 = (nonzero_vals.std() / nonzero_vals.mean()) ** 2
        else:
            cv2 = 0

    if adi < 1.32 and cv2 < 0.49:
        demand_class = 'smooth'
    elif adi >= 1.32 and cv2 < 0.49:
        demand_class = 'intermittent'
    elif adi < 1.32 and cv2 >= 0.49:
        demand_class = 'erratic'
    else:
        demand_class = 'lumpy'

    return {'adi': adi, 'cv2': cv2, 'class': demand_class}

In [ ]:
classifications = []
for item_id in df_long['id'].unique():
    item_sales = df_long[df_long['id'] == item_id]['sales'].values
    result = classify_demand(item_sales)
    result['id'] = item_id
    classifications.append(result)

class_df = pd.DataFrame(classifications)

print(f"\nРаспределение классов:\n{class_df['class'].value_counts()}")

## 1. Croston (1972) — оригинальный метод

**Идея:** разделить спрос на два компонента:
- **p** = интервал между ненулевыми продажами (периоды);
- **z** = размер ненулевого спроса.

Каждый прогнозируется отдельным SES:
- p̂ₜ = α·pₜ + (1−α)·p̂ₜ₋₁  (обновляется только при ненулевом спросе)
- ẑₜ = α·zₜ + (1−α)·ẑₜ₋₁  (обновляется только при ненулевом спросе)

Прогноз: ŷ = ẑ / p̂ (средний спрос за период).

In [ ]:
# Визуализация концепции на одном товаре
example_id = class_df[class_df['class'] == 'intermittent']['id'].iloc[0]
example_data = df_long[df_long['id'] == example_id].copy()
example_series = example_data['sales'].values[:200]

# Извлекаем интервалы и размеры
nonzero_idx = np.where(example_series > 0)[0]
intervals = np.diff(nonzero_idx)
sizes = example_series[nonzero_idx[1:]]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Исходный ряд
axes[0].plot(example_series, linewidth=0.8)
axes[0].scatter(nonzero_idx, example_series[nonzero_idx], color='red', s=20, zorder=5)
axes[0].set_ylabel('Продажи')
axes[0].set_title(f'Исходный ряд: {example_id}')
axes[0].grid(alpha=0.3)

# Интервалы
axes[1].plot(nonzero_idx[1:], intervals, 'o-', markersize=4)
axes[1].axhline(intervals.mean(), color='red', linestyle='--', label=f'Средний интервал p̂={intervals.mean():.1f}')
axes[1].set_ylabel('Интервал (периоды)')
axes[1].set_title('Интервалы между ненулевыми продажами')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Размеры
axes[2].plot(nonzero_idx[1:], sizes, 's-', markersize=4)
axes[2].axhline(sizes.mean(), color='red', linestyle='--', label=f'Средний размер ẑ={sizes.mean():.1f}')
axes[2].set_ylabel('Размер (ед.)')
axes[2].set_title('Размеры ненулевых продаж')
axes[2].set_xlabel('День')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Прогноз Croston: ŷ = ẑ/p̂ = {sizes.mean():.1f}/{intervals.mean():.1f} = {sizes.mean()/intervals.mean():.2f}")

### Смещение Croston

**Проблема:** ŷ = ẑ/p̂ — это отношение средних, а не среднее отношения → смещённая оценка.
При малых α смещение усиливается.

In [ ]:
# Демонстрация смещения на синтетике
np.random.seed(42)

# Генерация прерывистого ряда
n = 100
p_true = 5  # истинный интервал
z_true = 10  # истинный размер
true_demand_rate = z_true / p_true

synthetic = []
for i in range(n):
    if np.random.rand() < 1/p_true:
        synthetic.append(np.random.poisson(z_true))
    else:
        synthetic.append(0)

synthetic = np.array(synthetic)

In [ ]:
# Прогноз Croston руками с разными α
def croston_forecast(series, alpha=0.1):
    nonzero_idx = np.where(series > 0)[0]

    if len(nonzero_idx) < 2:
        return 0

    # Инициализация
    p_hat = np.diff(nonzero_idx)[0] if len(nonzero_idx) > 1 else len(series)
    z_hat = series[nonzero_idx[0]]

    # Обновления при каждом ненулевом спросе
    for idx in nonzero_idx[1:]:
        interval = idx - nonzero_idx[np.where(nonzero_idx < idx)[0][-1]]
        p_hat = alpha * interval + (1 - alpha) * p_hat

        z_hat = alpha * series[idx] + (1 - alpha) * z_hat

    return z_hat / p_hat

In [ ]:
alphas = [0.05, 0.1, 0.2, 0.5]
forecasts = [croston_forecast(synthetic, alpha=a) for a in alphas]

print(f"Истинный уровень спроса: {true_demand_rate:.2f}")
print("\nПрогноз Croston с разными α:")
for a, fc in zip(alphas, forecasts):
    bias = fc - true_demand_rate
    print(f"  α={a:.2f}: ŷ={fc:.2f}, смещение={bias:+.2f} ({100*bias/true_demand_rate:+.1f}%)")

print("\nМалый α → большее смещение (перепрогноз).")

## 2. SBA (Syntetos–Boylan Approximation)

Коррекция смещения: ŷ_SBA = (1 − α/2) · ŷ_Croston

Работает лучше при малых α.

In [ ]:
# Сравнение на том же синтетическом ряду
alpha = 0.1
croston_fc = croston_forecast(synthetic, alpha=alpha)
sba_fc = (1 - alpha/2) * croston_fc

print(f"Истинный уровень: {true_demand_rate:.2f}")
print(f"Croston (α={alpha}): {croston_fc:.2f}, ошибка {100*(croston_fc - true_demand_rate)/true_demand_rate:+.1f}%")
print(f"SBA (α={alpha}): {sba_fc:.2f}, ошибка {100*(sba_fc - true_demand_rate)/true_demand_rate:+.1f}%")

## 3. TSB (Teunter–Syntetos–Babai)

**Вероятностный подход:** моделируем вероятность ненулевого спроса отдельно от размера.
- ŷ = p̂ₜ · ẑₜ
- p̂ₜ = вероятность спроса (SES на индикаторе {yₜ > 0})
- ẑₜ = ожидаемый размер при ненулевом спросе (SES на ненулевых значениях)

Преимущество: работает лучше на коротких горизонтах.

In [ ]:
# TSB руками
def tsb_forecast(series, alpha_p=0.1, alpha_z=0.1):
    # Инициализация
    nonzero_idx = np.where(series > 0)[0]

    if len(nonzero_idx) == 0:
        return 0

    p_hat = (series > 0).mean()  # начальная вероятность
    z_hat = series[nonzero_idx].mean()  # начальный размер

    # Обновления
    for t in range(1, len(series)):
        # Вероятность
        indicator = 1 if series[t] > 0 else 0
        p_hat = alpha_p * indicator + (1 - alpha_p) * p_hat

        # Размер (только при ненулевом спросе)
        if series[t] > 0:
            z_hat = alpha_z * series[t] + (1 - alpha_z) * z_hat

    return p_hat * z_hat

In [ ]:
tsb_fc = tsb_forecast(synthetic, alpha_p=0.1, alpha_z=0.1)

print(f"Истинный уровень: {true_demand_rate:.2f}")
print(f"TSB: {tsb_fc:.2f}, ошибка {100*(tsb_fc - true_demand_rate)/true_demand_rate:+.1f}%")

## 4. ADIDA и IMAPA — агрегация временная

**ADIDA** (Aggregation–Disaggregation Intermittent Demand Approach):
- агрегируем ряд в несколько временных разрешений (день, неделя, месяц);
- прогнозируем на каждом уровне;
- дезагрегируем обратно и усредняем.

**IMAPA** (Intermittent Multiple Aggregation Prediction Algorithm):
- похоже на ADIDA, но с адаптивными весами по уровням агрегации.

In [ ]:
# Визуализация концепции ADIDA
example_daily = example_series[:200]

# Агрегация в недели
weekly = pd.Series(example_daily).rolling(7, min_periods=1).sum().values[::7]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

axes[0].plot(example_daily, linewidth=0.8)
axes[0].set_ylabel('Продажи (день)')
axes[0].set_title('Дневной уровень (прерывистый)')
axes[0].grid(alpha=0.3)

axes[1].plot(weekly, 'o-', linewidth=0.8)
axes[1].set_ylabel('Продажи (неделя)')
axes[1].set_title('Недельный уровень (более гладкий)')
axes[1].set_xlabel('Неделя')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("ADIDA усредняет прогнозы с разных уровней агрегации → робастность.")

## 5. Сравнение всех методов на M5 CA_1

Прогнозируем все товары, оцениваем метрики по классам спроса.

In [ ]:
# Подготовка данных
H = 28

train = df_long.groupby('id').apply(lambda x: x.iloc[:-H]).reset_index(drop=True)
test = df_long.groupby('id').apply(lambda x: x.iloc[-H:]).reset_index(drop=True)

train_sf = train[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})
test_sf = test[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})

print(f"Train: {len(train_sf)}, Test: {len(test_sf)}")

In [ ]:
# Модели специализированные
models = [
    Croston(),
    CrostonOptimized(),
    CrostonSBA(),
    TSB(alpha_d=0.1, alpha_p=0.1),
    ADIDA(),
    IMAPA()
]

sf = StatsForecast(models=models, freq='D', n_jobs=1)

In [ ]:
# Прогноз
sf.fit(train_sf)
forecast = sf.predict(h=H)

In [ ]:
# Добавляем класс спроса
forecast = forecast.reset_index().merge(class_df[['id', 'class']], left_on='unique_id', right_on='id', how='left')
test_sf = test_sf.merge(class_df[['id', 'class']], left_on='unique_id', right_on='id', how='left')

In [ ]:
# Метрики
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(((y_true - y_pred) ** 2).mean())
    mae = np.abs(y_true - y_pred).mean()
    bias = (y_true - y_pred).mean()
    return {'RMSE': rmse, 'MAE': mae, 'Bias': bias}

In [ ]:
results = []

model_names = ['Croston', 'CrostonOptimized', 'CrostonSBA', 'TSB', 'ADIDA', 'IMAPA']

for model_name in model_names:
    for cls in ['smooth', 'intermittent', 'erratic', 'lumpy', 'all']:
        if cls == 'all':
            test_cls = test_sf
            fc_cls = forecast
        else:
            test_cls = test_sf[test_sf['class'] == cls]
            fc_cls = forecast[forecast['class'] == cls]

        if len(test_cls) == 0:
            continue

        y_true_all = []
        y_pred_all = []

        for uid in test_cls['unique_id'].unique():
            yt = test_cls[test_cls['unique_id'] == uid]['y'].values
            yp = fc_cls[fc_cls['unique_id'] == uid][model_name].values

            if len(yt) == len(yp):
                y_true_all.extend(yt)
                y_pred_all.extend(yp)

        if len(y_true_all) > 0:
            metrics = compute_metrics(np.array(y_true_all), np.array(y_pred_all))
            results.append({
                'Модель': model_name,
                'Класс': cls,
                **metrics,
                'Товаров': test_cls['unique_id'].nunique()
            })

results_df = pd.DataFrame(results)

In [ ]:
# Визуализация RMSE по классам
pivot = results_df[results_df['Класс'] != 'all'].pivot(index='Модель', columns='Класс', values='RMSE')

pivot.plot(kind='bar', figsize=(12, 5))
plt.ylabel('RMSE')
plt.title('Специализированные методы: RMSE по классам спроса')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Класс спроса')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("\nМетрики по классам спроса:\n")
print(results_df[results_df['Класс'] != 'all'][['Модель', 'Класс', 'RMSE', 'MAE', 'Bias']].to_string(index=False))

## 6. Когда какой метод

| Метод | Лучше работает для | Недостатки |
|-------|-------------------|-----------|
| Croston | Intermittent (ADI > 1.32, CV² < 0.49) | Смещение при малых α |
| SBA | Intermittent, коррекция смещения | Всё ещё не идеален при lumpy |
| TSB | Короткие горизонты, вероятностная интерпретация | Сложнее настройка |
| ADIDA | Lumpy, нестабильный спрос | Требует достаточно истории |
| IMAPA | Адаптивность по уровням агрегации | Вычислительно дороже |

**Общий вывод:** нет универсального победителя, но TSB и SBA обычно лучше оригинального Croston.

## Итог

- Croston разделяет прогноз на интервалы и размеры → хорошо для intermittent.
- SBA корректирует смещение Croston → улучшение на 5-15%.
- TSB моделирует вероятность ненулевого спроса → лучше на коротких горизонтах.
- ADIDA/IMAPA используют агрегацию → робастность к выбросам и lumpy.
- На M5 данных TSB и IMAPA показывают лучшие результаты для intermittent/lumpy классов.

**Следующий ноутбук:** современные подходы — бутстреп для прерывистого спроса, zero-inflated модели, нейронные методы.